# Задание: LSTM-классификатор отзывов

Сборная Москвы по ИИ | NLP, занятие 1

---

На семинаре мы познакомились с подходами Bag of Words, TF-IDF и Word2Vec. Все они теряют информацию о **порядке слов** — «не хороший фильм» и «хороший не фильм» для них одно и то же.

В этом задании вы реализуете **LSTM-классификатор**, который обрабатывает текст последовательно и способен улавливать контекст.

**Структура:**
- Задание 1. Реализовать `SentimentLSTM`
- Задание 2. Обучить LSTM и проанализировать результат
- Задание 3. Выводы
- Бонус. Предобученные GloVe-эмбеддинги

**Правила:**
- Заполняйте ячейки с пометкой `# TODO`
- Не удаляйте существующий код
- Комментируйте свои решения

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

import nltk
from nltk.tokenize import word_tokenize

from sklearn.metrics import accuracy_score

from datasets import load_dataset

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## Загрузка и подготовка данных (дано)

Загружаем IMDB (25k train / 25k test). Для ускорения работы берём подвыборку.

In [ ]:
dataset = load_dataset('imdb')

N_TRAIN = 10000
N_TEST = 5000

np.random.seed(42)
train_idx = np.random.choice(len(dataset['train']), N_TRAIN, replace=False)
test_idx = np.random.choice(len(dataset['test']), N_TEST, replace=False)

train_raw = [dataset['train'][int(i)]['text'] for i in train_idx]
test_raw = [dataset['test'][int(i)]['text'] for i in test_idx]
y_train = np.array([dataset['train'][int(i)]['label'] for i in train_idx])
y_test = np.array([dataset['test'][int(i)]['label'] for i in test_idx])

print(f"Train: {len(train_raw)}, Test: {len(test_raw)}")
print(f"Метки (train): {Counter(y_train)}")
print(f"Метки (test):  {Counter(y_test)}")

In [ ]:
train_tokens = [word_tokenize(t.lower()) for t in train_raw]
test_tokens = [word_tokenize(t.lower()) for t in test_raw]

print(f"Средняя длина отзыва: {np.mean([len(t) for t in train_tokens]):.0f} токенов")

---
## Словарь и Dataset (дано)

Строим словарь из тренировочных данных (только слова с частотой >= 5), создаём Dataset и DataLoader с паддингом.

In [ ]:
PAD_IDX = 0
UNK_IDX = 1
MIN_FREQ = 5
MAX_SEQ_LEN = 256

word_counts = Counter()
for tokens in train_tokens:
    word_counts.update(tokens)

vocab = {'<pad>': PAD_IDX, '<unk>': UNK_IDX}
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print(f"Размер словаря: {len(vocab)}")

In [ ]:
def tokens_to_ids(tokens, vocab, max_len=MAX_SEQ_LEN):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    return ids


class IMDBDataset(Dataset):
    def __init__(self, token_lists, labels, vocab):
        self.data = [torch.tensor(tokens_to_ids(t, vocab), dtype=torch.long) for t in token_lists]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


def collate_fn(batch):
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts])
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return texts_padded, torch.stack(labels), lengths


train_ds = IMDBDataset(train_tokens, y_train, vocab)
test_ds = IMDBDataset(test_tokens, y_test, vocab)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn)

print(f"Пример батча:")
texts_b, labels_b, lengths_b = next(iter(train_loader))
print(f"  texts: {texts_b.shape}, labels: {labels_b.shape}, lengths: {lengths_b[:5]}")

---
## Задание 1. Реализовать `SentimentLSTM`

Напишите LSTM-модель для бинарной классификации текста.

Архитектура:
1. `nn.Embedding(vocab_size, emb_dim, padding_idx=0)` — превращает индексы токенов в плотные векторы
2. `nn.LSTM(emb_dim, hidden_dim, batch_first=True)` — рекуррентный слой, обрабатывающий последовательность
3. `nn.Linear(hidden_dim, num_classes)` — линейный классификатор поверх последнего hidden state

Рекомендуемые параметры: `emb_dim=64`, `hidden_dim=128`.

Подсказка для `forward`:
- Получите эмбеддинги: `emb = self.embedding(x)` → `[batch, seq_len, emb_dim]`
- Пропустите через LSTM: `output, (h_n, c_n) = self.lstm(emb)` → `h_n` имеет форму `[1, batch, hidden_dim]`
- Возьмите `h_n[-1]` (последний hidden state) и подайте в `Linear`

Обратите внимание: на семинаре мы строили LSTM для **генерации** текста (предсказание следующего символа). Здесь задача другая — **классификация** целого текста. Ключевое отличие: вместо того чтобы выдавать предсказание на каждом шаге, мы берём **итоговое** скрытое состояние LSTM (после обработки всей последовательности) и подаём его в линейный слой.

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=128, num_classes=2):
        super().__init__()
        # TODO: определите слои (Embedding, LSTM, Linear)
        pass

    def forward(self, x, lengths=None):
        # TODO: определите forward pass
        # x: [batch, seq_len] — индексы токенов
        # Верните логиты [batch, num_classes]
        pass

---
## Задание 2. Обучение и анализ

**2.1** Создайте модель, оптимизатор (`Adam`, `lr=1e-3`) и функцию потерь (`CrossEntropyLoss`).

In [ ]:
# TODO: создайте модель, оптимизатор и функцию потерь

model = ...  # TODO: SentimentLSTM(len(vocab), ...).to(device)
optimizer = ...  # TODO
criterion = ...  # TODO

print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

**2.2** Напишите цикл обучения на 5–10 эпох.

Для каждой эпохи:
1. **Train**: для каждого батча сделайте `forward → loss → backward → step`
2. **Eval**: посчитайте accuracy на тесте

Выведите loss и test accuracy на каждой эпохе.

Напоминание: `DataLoader` возвращает `(texts_padded, labels, lengths)` — не забудьте перенести тензоры на `device`.

In [ ]:
n_epochs = 7
train_losses, test_accs = [], []

for epoch in range(n_epochs):
    # --- Train ---
    model.train()
    epoch_loss = 0
    for texts_batch, labels_batch, lengths_batch in train_loader:
        texts_batch = texts_batch.to(device)
        labels_batch = labels_batch.to(device)

        # TODO: forward, loss, backward, step
        pass

    # --- Eval ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for texts_batch, labels_batch, lengths_batch in test_loader:
            texts_batch = texts_batch.to(device)
            labels_batch = labels_batch.to(device)

            # TODO: forward, подсчёт правильных ответов
            pass

    test_acc = correct / total if total > 0 else 0
    test_accs.append(test_acc)
    print(f"Epoch {epoch+1}/{n_epochs} | Test accuracy: {test_acc:.3f}")

best_acc = max(test_accs) if test_accs else 0
print(f"\nЛучшая test accuracy: {best_acc:.3f}")

**2.3** Постройте графики обучения (loss и accuracy по эпохам).

In [ ]:
# TODO: постройте два графика рядом (plt.subplots(1, 2)):
# 1) Train Loss по эпохам
# 2) Test Accuracy по эпохам

---
## Задание 3. Выводы

Напишите 3–5 предложений:
- Какой accuracy получился у LSTM? Сравните с результатами семинара (TF-IDF + LogReg ~0.88, Word2Vec + MLP ~0.85).
- Почему LSTM может работать лучше (или хуже), чем подходы с усреднением (BoW / Word2Vec + mean)?
- Какие недостатки у LSTM по сравнению с более простыми методами (скорость, ресурсы, сложность)?

**Ваши выводы:**

*TODO: напишите здесь*

---
## Бонус. Предобученные GloVe-эмбеддинги

LSTM из основного задания учит эмбеддинги с нуля на 10k отзывов — это мало. Существуют эмбеддинги, обученные на **миллиардах** слов (GloVe, fastText), которые уже «знают» семантику.

Задача:
1. Загрузите предобученные GloVe-векторы (100d)
2. Постройте матрицу эмбеддингов для вашего словаря
3. Инициализируйте `Embedding` слой этой матрицей
4. Обучите LSTM заново и сравните accuracy

```python
import gensim.downloader as api
glove = api.load('glove-wiki-gigaword-100')
```

Подсказка: для инициализации используйте `model.embedding.weight.data.copy_(embedding_matrix)`.

In [ ]:
# TODO (бонус): загрузите GloVe, постройте матрицу эмбеддингов для вашего словаря,
# инициализируйте Embedding слой и обучите LSTM заново.
# Сравните accuracy с LSTM без предобученных эмбеддингов.